# 03 — Découpage, alignement des faits et budget de longueur

**Projet** : Résumé de comptes-rendus d'intervention — encodeur-décodeur appris, baseline extractive publiée

## Objectifs pédagogiques

1. Découper un compte-rendu en phrases et en tokens, et vérifier que rien ne se perd — les faits annotés sont repérés au caractère.
1. Calculer le budget de longueur qu'un modèle respectera, et mesurer la compression réellement demandée par le corpus.
1. Mesurer ce qu'un **extracteur de phrases** retrouverait déjà : le plafond de la recopie, avant toute réécriture.
1. Vérifier si la centralité d'une phrase suffit à prédire sa saillance — la réponse conditionne le choix d'un modèle extractif ou génératif.

## 0. Environnement et corpus

Le notebook lit son propre corpus réduit (`outputs/notebooks/data`) ; s'il manque, il le génère.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (200 comptes-rendus), un budget
# d'époques réduit et une architecture réduite : l'exécution reste de quelques secondes, et les
# valeurs absolues d'un notebook ne sont **pas** celles de la référence publiée par le README.
NB_DOCUMENTS = 200
NB_EPOCHS = 8

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "model.params.layers=1",
                "model.params.units=96",
                "model.params.pretraining.epochs=1",
                "+model.params.vocab_size=800",
                "+model.params.min_frequency=2",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
                "train.early_stopping.enabled=false",
                "seed=42",
                "log_level=WARNING",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Bac à sable : les notebooks génèrent **leur** corpus (réduit) et écrivent **leurs** artefacts sous
# `outputs/notebooks/`. `data/raw`, `artifacts/` et donc les chiffres publiés par le README restent
# ceux de `make data` / `make train` : aucun notebook ne peut les modifier par accident.
NB_ROOT = PROJECT_ROOT / "outputs" / "notebooks"
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=NB_ROOT / "data",
    artifacts_dir=NB_ROOT / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
ID_COLUMN = str(MODEL_NODE.get("id_column", "doc_id"))
COMPRESSION = float(MODEL_NODE.get("params", {}).get("compression", 0.45))
MAX_OUTPUT_TOKENS = int(MODEL_NODE.get("params", {}).get("max_output_tokens", 90))
STRATEGIES = list(MODEL_NODE.get("strategies_to_compare", []) or ["lead"])

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.45)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Stratégies comparées : {', '.join(STRATEGIES)}")
print(f"Budget de longueur: compression {COMPRESSION:.2f}, plafond {MAX_OUTPUT_TOKENS} tokens")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticSummaryCorpusGenerator
from src.data.loaders import SummaryCorpusLoader

LOADER = SummaryCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS, REFERENCES, FACTS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(
        f"Corpus du notebook lu : {len(DOCUMENTS)} comptes-rendus, "
        f"{len(REFERENCES)} résumés de référence, {len(FACTS)} faits annotés"
    )
else:
    # Un clone frais n'a pas encore de corpus réduit : le notebook le génère, de façon déterministe,
    # dans son bac à sable. `make data` produit le corpus de référence (600 comptes-rendus) dans
    # `data/raw` — les deux ne se mélangent jamais.
    bundle = SyntheticSummaryCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, REFERENCES, FACTS = bundle.documents, bundle.queries, bundle.facts
    METADATA = bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_references(REFERENCES)
    LOADER.save_facts(FACTS)
    LOADER.save_metadata(METADATA)
    print(f"Corpus réduit généré dans outputs/notebooks/data : {len(DOCUMENTS)} comptes-rendus")

TRAIN_DOCUMENTS, TRAIN_REFERENCES, TRAIN_FACTS = LOADER.load_split("train")
VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS = LOADER.load_split("val")
TEST_DOCUMENTS, TEST_REFERENCES, TEST_FACTS = LOADER.load_split("test")
SPLIT_SIZES = LOADER.split_sizes()
# Le test ne sert **jamais** à régler quoi que ce soit : les réglages se lisent sur la validation,
# le verdict sur le test, une seule fois.
GOLDEN = dict(zip(REFERENCES["doc_id"], REFERENCES["summary"], strict=False))
print("splits :", SPLIT_SIZES)
print("compression moyenne demandée :", round(float(REFERENCES["compression"].mean()), 4))

## 1. Découper un compte-rendu

Un fait annoté est repéré par une **surface exacte** dans le texte et par l'index de sa phrase. Toute étape de prétraitement qui déplace le texte (normalisation, retrait de doublons) déplace donc aussi les annotations : c'est la contrainte qui décide de qui a le droit de réécrire le corpus.

In [ ]:
from src.preprocessing.transformers import normalise_text, split_sentences, tokenize

example_id = str(DOCUMENTS.iloc[0]["doc_id"])
example_text = DOCUMENTS.loc[DOCUMENTS["doc_id"] == example_id, "text"].iloc[0]
sentences = split_sentences(example_text)
sentence_table = pd.DataFrame(
    {
        "phrase": range(len(sentences)),
        "tokens": [len(tokenize(sentence)) for sentence in sentences],
        "caractères": [len(sentence) for sentence in sentences],
        "début": [example_text.find(sentence) for sentence in sentences],
        "extrait": [sentence[:70] for sentence in sentences],
    }
)
display(sentence_table)

declared_sentences = int(DOCUMENTS.loc[DOCUMENTS["doc_id"] == example_id, "n_sentences"].iloc[0])
print("phrases reconstituées :", len(sentences), "| déclarées au corpus :", declared_sentences)
print("tokénisation sans perte  :", len(tokenize(example_text)), "tokens")
print("normalisation (exemple)  :", normalise_text("Le  Câble   s'est DÉTENDU !"))

**Ce qu'il faut retenir**

- Le nombre de phrases reconstituées doit égaler la valeur déclarée par le corpus : un écart signalerait un découpage différent de celui qui a produit les annotations.
- L'offset de chaque phrase est publié (`début`) : c'est lui qui permet de rattacher une surface de fait à sa phrase, et donc de vérifier l'annotation au lieu de la supposer.
- La tokénisation est sans perte par construction : elle sert à **mesurer**, pas à réécrire le corpus. Une normalisation destructive casserait tous les offsets.

### 1.1 Les faits sont alignés sur le texte

Trois vérifications, sur tout le corpus : la surface est dans le document, la valeur est portée par la phrase annoncée, et l'index de phrase existe.

In [ ]:
texts = dict(zip(DOCUMENTS["doc_id"], DOCUMENTS["text"], strict=False))
out_of_bounds = 0
surface_missing = 0
sentence_mismatch = 0
for row in FACTS.itertuples(index=False):
    document = texts[str(row.doc_id)]
    if str(row.surface) not in document:
        surface_missing += 1
        continue
    document_sentences = split_sentences(document)
    if int(row.sentence_index) >= len(document_sentences):
        out_of_bounds += 1
        continue
    if (
        str(row.value).split()[0].casefold()
        not in document_sentences[int(row.sentence_index)].casefold()
    ):
        sentence_mismatch += 1

print(f"faits hors bornes            : {out_of_bounds}")
print(f"surfaces absentes du document: {surface_missing}")
print(f"valeurs absentes de la phrase annoncée : {sentence_mismatch}")
example_facts = FACTS[FACTS["doc_id"] == example_id]
display(example_facts[["fact_type", "value", "surface", "sentence_index", "salient"]])

**Ce qu'il faut retenir**

- Un fait mal aligné ne casse aucun contrat de table : il fausse silencieusement la couverture, parce qu'un modèle ne pourra jamais retrouver une surface qui n'est pas dans le texte.
- Les faits non saillants sont conservés dans la table : ils ne sont pas mesurés (le résumé n'a pas à les rapporter) mais ils rendent visible ce qu'un modèle qui recopie tout gagnerait — rien, en couverture.
- La phrase annoncée est vérifiée séparément de la surface : une valeur peut être dans le document **et** dans la mauvaise phrase si l'annotation a été décalée.

## 2. Le budget de longueur

Le modèle ne décide pas librement de sa longueur : il reçoit un budget, proportionnel au document et borné par un plancher et un plafond. Le notebook recalcule ce budget comme le fera le modèle, puis le compare aux résumés de référence.

In [ ]:
from src.data.vocabulary import MAX_COMPRESSION
from src.training.metrics import length_stats

# Reconstitution explicite de la formule du contrat : arrondi de `compression x tokens`, borné par
# `min_output_tokens` puis par `max_output_tokens`. La recalculer ici est le but de la cellule : un
# budget qu'on ne sait pas recalculer est un budget qu'on ne sait pas expliquer.
MIN_OUTPUT_TOKENS = int(node(CONFIG, "model").get("params", {}).get("min_output_tokens", 12))
computed = (DOCUMENTS["n_tokens"] * COMPRESSION).round().clip(MIN_OUTPUT_TOKENS, MAX_OUTPUT_TOKENS)
budget_table = pd.DataFrame({"document": DOCUMENTS["n_tokens"], "budget": computed})
budget_table["référence"] = REFERENCES["n_tokens"].to_numpy()
display(budget_table.describe().round(2))

print("budget moyen                 :", round(float(computed.mean()), 1), "tokens")
print("longueur moyenne des résumés :", round(float(REFERENCES["n_tokens"].mean()), 1), "tokens")
print(
    "résumés au-dessus du budget  :",
    int((REFERENCES["n_tokens"] > computed).sum()),
    "/",
    len(REFERENCES),
)
print(
    "part des résumés qui atteignent le plafond du budget  :",
    round(float((REFERENCES["compression"] >= MAX_COMPRESSION - 0.001).mean()), 3),
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(budget_table["budget"], bins=30, alpha=0.6, label="budget du modèle")
axes[0].hist(budget_table["référence"], bins=30, alpha=0.6, label="résumé de référence")
axes[0].set_title("Budget et longueur des références (tokens)")
axes[0].legend()
axes[1].scatter(budget_table["document"], budget_table["référence"], s=8, alpha=0.5)
axes[1].set_xlabel("tokens du document")
axes[1].set_ylabel("tokens du résumé de référence")
axes[1].set_title("Compression observée")
plt.tight_layout()
plt.show()

# La table des références ne porte pas les colonnes d'une **prédiction** : on les construit
# ici, avec le budget recalculé plus haut, plutôt que d'ajouter de fausses colonnes.
reference_lengths = pd.DataFrame(
    {
        "compression": REFERENCES["compression"].to_numpy(),
        "n_sentences": REFERENCES["n_sentences"].to_numpy(),
        "hit_max_length": (REFERENCES["n_tokens"].to_numpy() >= computed.to_numpy()).astype(int),
    }
)
print("longueurs de référence, mesurées avec les métriques du projet :")
display(pd.Series(length_stats(reference_lengths), name="référence").round(4))

**Ce qu'il faut retenir**

- Un budget calculé **avant** l'entraînement évite le réglage a posteriori : le modèle apprend à tenir dans une longueur, il ne la découvre pas à l'inférence.
- Si la plupart des résumés de référence dépassaient le budget, le rappel de ROUGE serait plafonné par construction — c'est le premier chiffre à regarder avant d'accuser le modèle.
- Le plancher et le plafond du budget ne sont pas cosmétiques : sans plancher, un document court produit un résumé de trois mots ; sans plafond, un document long produit un décodeur qui n'en finit pas.

## 3. Ce qu'un extracteur de phrases retrouverait déjà

Avant de générer, on mesure la **recopie** : pour chaque phrase du résumé de référence, quelle est la meilleure phrase du document ? Ce plafond d'extraction dit la part de la tâche qui n'est qu'une sélection, et celle qui exige une reformulation.

In [ ]:
from src.evaluation.rouge import rouge_scores

SAMPLE = VAL_DOCUMENTS.head(40)
golden = dict(zip(VAL_REFERENCES["doc_id"], VAL_REFERENCES["summary"], strict=False))
oracle_scores = []
for row in SAMPLE.itertuples(index=False):
    document_sentences = split_sentences(str(row.text))
    reference_sentences = split_sentences(golden[str(row.doc_id)])
    for reference_sentence in reference_sentences:
        best = max(
            (
                rouge_scores(reference_sentence, candidate).rouge1.f1
                for candidate in document_sentences
            ),
            default=0.0,
        )
        oracle_scores.append(best)

oracle = pd.Series(oracle_scores, name="meilleure phrase du document (ROUGE-1)")
print("plafond d'extraction (recopier la meilleure phrase par phrase de référence) :")
print(f"  ROUGE-1 moyen : {oracle.mean():.4f}")
print(
    f"  phrases du résumé parfaitement retrouvables (ROUGE-1 = 1,0) : "
    f"{float((oracle >= 0.999).mean()):.1%}"
)
print(
    f"  phrases sans aucune correspondance correcte (ROUGE-1 < 0,5) : "
    f"{float((oracle < 0.5).mean()):.1%}"
)

**Ce qu'il faut retenir**

- Un extracteur de phrases est plafonné par ce chiffre : au-delà, il faut réécrire. C'est le même argument que la comparaison des stratégies, mais mesuré sur la référence au lieu d'un modèle.
- Les phrases sans correspondance correcte sont celles qui portent une **synthèse** (ou une reformulation) : elles justifient un modèle génératif, à condition que la génération reste fidèle.
- Mesurer le plafond sur la validation, jamais sur le test : le test ne sert qu'une fois, pour le verdict du notebook 06.

## 4. La centralité suffit-elle ?

Une heuristique classique prend les phrases **les plus centrales** (proches des autres). Le projet construit ce graphe, donc la question se mesure : les phrases centrales portent-elles les faits saillants ?

In [ ]:
from src.features.build_features import SentenceFeatureBuilder

builder = SentenceFeatureBuilder.from_config(CONFIG.preprocessing).build(example_id, example_text)
print(builder.summary())
central = builder.most_central_sentences(3)
for rank, (index, score) in enumerate(central, start=1):
    print(f"  {rank}. phrase {index} (centralité {score:.4f}) — {builder.sentences[index][:80]}")
    print("     termes saillants :", [term for term, _ in builder.top_terms(index, 5)])

In [ ]:
hits = []
for row in VAL_DOCUMENTS.head(40).itertuples(index=False):
    features = SentenceFeatureBuilder.from_config(CONFIG.preprocessing).build(
        str(row.doc_id), str(row.text)
    )
    central_indices = [index for index, _ in features.most_central_sentences(3)]
    salient = FACTS[(FACTS["doc_id"] == str(row.doc_id)) & (FACTS["salient"] == 1)]
    if salient.empty:
        continue
    hits.append(float(salient["sentence_index"].isin(central_indices).mean()))

centrality_hit = pd.Series(hits, name="faits saillants dans les 3 phrases les plus centrales")
print(f"documents mesurés : {len(centrality_hit)}")
print(f"part des faits saillants portés par une phrase centrale : {centrality_hit.mean():.1%}")
print(
    f"documents où toutes les phrases centrales portent un fait : "
    f"{float((centrality_hit > 0).mean()):.1%}"
)

**Ce qu'il faut retenir**

- Si la centralité retrouvait la majorité des faits saillants, une baseline extractive suffirait — et le projet publierait cette baseline comme référence, ce qu'il fait (``textrank``).
- La centralité est lexicale : elle ne distingue pas « la pièce CRR-255 » de « la presse PR-6 ». Un fait rare pèse moins qu'un mot fréquent, donc il passe sous le seuil de sélection.
- C'est exactement l'écart de mesure entre ROUGE (ressemblance aux mots de la référence) et couverture (présence des faits annotés) : les deux sont publiés.